# HPA Data Access with `pyVO` 

```
Author: ESDC Team at ESAC
Date Last Modified: 20/08/2026
```

This notebook demonstrate programmatic access to the data in ESA's [HPA](https://hpa.esa.int) using the Table Access Protocol (TAP) and the Astronomical Query Language (ADQL) with the [PyVO](https://pyvo.readthedocs.io/en/latest/) `python` package. Please refer to the [PyVO documentation](https://pyvo.readthedocs.io/en/stable/dal/index.html#pyvo-scs) for a more in-depth introduction.

## Requirements

The following python packages are required to run this notebook: `pyvo`. You can install them by running the line below in your terminal (after removing the leading `#`).

In [ ]:
# python -m pip install pyvo

In [2]:
import pyvo as vo

## Connecting to the HPA via TAP

To connect to the HPA, we use the TAP service protocol. This requires a TAP service URL, which is provided by the HPA in the documentation pages of the different archives. We use here the URL of the Proba-2 mission archive.

In [3]:
URL_TAP = "https://p2sa.esac.esa.int/p2sa-sl-tap/tap"

tap_service = vo.dal.TAPService(URL_TAP)

## Inspecting table metadata

The Table Acess Protocol exposes different tables to the user. We can inspect the tables present in a serivce using the `tables` attribute of the `TAPService` object. Each `table` attribute further contains information on the columns and datatypes it provides.

In [18]:
table_names = [table.name for table in tap_service.tables]
print(len(tap_service.tables), "tables found in the TAP service: ", table_names)

21 tables found in the TAP service:  ['p2sa.file', 'p2sa.full_disk_solar_image', 'p2sa.instrument', 'p2sa.lyra_observation', 'p2sa.observation', 'p2sa.observatory', 'p2sa.science_object', 'p2sa.swap_observation', 'p2sa.v_carrington_rotation_file', 'p2sa.v_file', 'p2sa.v_lyra_observation', 'p2sa.v_observation', 'p2sa.v_swap_observation', 'public.dual', 'tap_config.coord_sys', 'tap_config.properties', 'tap_schema.columns', 'tap_schema.key_columns', 'tap_schema.keys', 'tap_schema.schemas', 'tap_schema.tables']


To inspect the columns of a specific table, you can use the following code:

In [21]:
table_observation = tap_service.tables['p2sa.observation']

for column in table_observation.columns:
    print(f"  {column.name} ({column.datatype.content})")

  begin_date (char)
  calibrated (boolean)
  end_date (char)
  file_format (char)
  file_name (char)
  file_path (char)
  file_size (long)
  instrument_oid (int)
  observation_oid (int)
  observation_type (char)
  processing_level (char)
  science_objective (char)
  science_object_oid (int)
  wavelength_range (char)


This table contains metadata of Proba-2 observations, including start and end dates, calibration levels, filenames, and instrument IDs. In the P2Sa, there is also a `v_observation` table. To see how it differs, we compare the columns between the two tables.  

In [22]:
columns_observation = [column.name for column in tap_service.tables['p2sa.observation'].columns]
columns_v_observation = [column.name for column in tap_service.tables['p2sa.v_observation'].columns]

difference = set(columns_v_observation) - set(columns_observation)
print("Columns in p2sa.v_observation but not in p2sa.observation:", difference)

Columns in p2sa.v_observation but not in p2sa.observation: {'observatory_name', 'science_object_name', 'instrument_name'}



We see that `v_observation` is slightly more human-readable than `observation`, as it contains the instrument, observatory, and science object names as well as their numeric IDs.

## Accessing table data

We now want to get the content of the tables themselves. This can be achieved via *synchronous* or *asynchronous* AQDL queries. The PyVO documentation contains a nice [summary of the pros and cons](https://pyvo.readthedocs.io/en/stable/dal/index.html#synchronous-vs-asynchronous-query) of each, but in short:

- Synchronous queries return the result immediately and generally do not require a login. They are best for small (in terms of data volume) queries.
- Asynchronous queries first compute and prepare the query result on the remote (HPA) side, before providing the user with a download link. These queries support much larger data volume but may require you to log in to an archive user account prior to querying.

### Synchronous table data access

All data queries are written in ADQL syntax. Extensive documentation for this query language is available [here](https://www.ivoa.net/documents/ADQL/20180112/PR-ADQL-2.1-20180112.html). To run a synchronous query with `pyVO`, use the `search` method of the `TAPService` object.

In [24]:
results = tap_service.search("select top 100 * from p2sa.observation")
print(results)

<DALResultsTable length=100>
       begin_date       calibrated ... science_object_oid wavelength_range
         object            bool    ...       int32             object     
----------------------- ---------- ... ------------------ ----------------
 2010-05-10T14:31:21.53       True ...                  1              174
 2010-05-11T23:04:47.18      False ...                  1              174
2010-05-10T11:11:21.364       True ...                  1              174
2010-05-11T19:38:46.968      False ...                  1              174
2010-05-11T16:48:46.823      False ...                  1              174
 2010-05-11T13:52:46.68      False ...                  1              174
2010-05-11T12:48:46.628      False ...                  1              174
2010-05-11T10:37:06.521      False ...                  1              174
2010-05-11T09:49:42.482      False ...                  1              174
                    ...        ... ...                ...              

This list contains the first 100 entries of the Proba-2 science archive observations table. You can convert it from the custom `DALResultsTable` type to the more common `Astropy.Table` or `Pandas.DataFrame` types.

In [25]:
results = results.to_table() # convert the results to an astropy table
results = results.to_pandas() # convert the astropy table to a pandas dataframe

Some services limit the number of rows that are returned in a single synchronous query. These limits are available via the `maxrec` and `hardlimit` attributes of the `TAPService`. `maxrec` can be changed by the user (refer to the [PyVO documentation](https://pyvo.readthedocs.io/en/stable/dal/index.html#query-limit)), but it cannot exceed `hardlimit`. For the P2SA, there are no limits in place, hence, looking up the attributes raises an error:

In [29]:
try:
    print("The set record limit is:", tap_service.maxrec)
except:
    print("The set record limit is not available for this TAP service.")

try:
    print("The maximum record limit is:", tap_service.hardlimit)
except:
    print("The maximum record limit is not available for this TAP service.")

The set record limit is not available for this TAP service.
The maximum record limit is not available for this TAP service.


Even though there is no limit in place, requesting the table of all observations in the `p2sa.observations` table via a synchronous query will likely not work, as the server connection will be lost before the query can finish. Large queries need to be run asynchronously.

### Asynchronous table data access

The `run_async` method of the `TAPService` object allows to execute ADQL queries asynchronously with almost the same syntax as for synchronous queries (only replacing `search` with `run_async`).

In [30]:
results = tap_service.run_async("select * from p2sa.observation")
print(results)

<DALResultsTable length=7475989>
       begin_date       calibrated ... science_object_oid wavelength_range
         object            bool    ...       int32             object     
----------------------- ---------- ... ------------------ ----------------
2014-03-20T20:02:03.484       True ...                  1              174
2015-03-18T20:31:13.593       True ...                  1              174
2015-03-18T21:03:13.627       True ...                  1              174
2015-03-18T22:37:13.726       True ...                  1              174
 2015-03-18T22:59:13.75       True ...                  1              174
2014-03-21T00:41:33.823      False ...                  1              174
2015-03-19T00:25:13.841      False ...                  1              174
2014-03-18T06:47:19.767       True ...                  1              174
2015-03-19T02:07:13.948      False ...                  1              174
                    ...        ... ...                ...          

Executing the query above returned 7475989 total observations and finished in about 5 minutes.


Under the hood, `run_async` submits a job to the TAP service, queries the job state intermittently, and retrieves the result once the job is completed. All these steps can also be executed manually using `pyVO`, as documented [here](https://pyvo.readthedocs.io/en/stable/dal/index.html#synchronous-vs-asynchronous-query). This gives a greater degree of flexbility (e.g. by providing progress information) by is slightly more verbose. 

#### Authentification

TBD.